In [1]:
# Cell 1: Team Selection UI
import json
import ipywidgets as widgets
from IPython.display import display

# Load eligible players
with open('eligible_players.json') as f:
    players = json.load(f)['players']

# Prepare position options, DH includes all players
positions = ['C', '1B', '2B', '3B', 'SS'] + ['OF'] * 3 + ['DH']
pos_options = {pos: ['Select Player'] for pos in set(positions)}
for p in players:
    pos = p['pos']
    if pos in pos_options and pos != 'DH':
        pos_options[pos].append(p['longName'])
    # Always include in DH
    pos_options['DH'].append(p['longName'])

bench_opts = ['Select Player'] + [p['longName'] for p in players if p['pos'] != 'P']

# Structure to hold selected players
enum_selected = {
    'C': {}, '1B': {}, '2B': {}, '3B': {}, 'SS': {},
    'OF': [{}, {}, {}], 'DH': {}, 'PH': []
}

# Create dropdown widgets for positions
pos_dropdowns = [widgets.Dropdown(options=pos_options[pos], description=pos) for pos in positions]
ph_dropdowns = [widgets.Dropdown(options=bench_opts, description=f'PH{i+1}') for i in range(6)]
select_btn = widgets.Button(description='Select team')
select_out = widgets.Output()

def select_team(b):
    select_out.clear_output()
    choices = [d.value for d in pos_dropdowns] + [d.value for d in ph_dropdowns]
    if 'Select Player' in choices:
        with select_out:
            print('Error: All slots must be filled.')
        return
    if len(choices) != len(set(choices)):
        with select_out:
            print('Error: Duplicate player selected.')
        return
    # Populate enum_selected
    of_idx = 0
    enum_selected['PH'].clear()
    for d in pos_dropdowns:
        name = d.value
        pid = next(p['playerID'] for p in players if p['longName'] == name)
        position = d.description
        if position == 'OF':
            enum_selected['OF'][of_idx] = {'id': pid, 'name': name}
            of_idx += 1
        else:
            enum_selected[position] = {'id': pid, 'name': name}
    for i, d in enumerate(ph_dropdowns):
        name = d.value
        pid = next(p['playerID'] for p in players if p['longName'] == name)
        enum_selected['PH'].append({'id': pid, 'name': name, 'bench_order': i+1})
    with select_out:
        print('Selected players:')
        print(enum_selected)

select_btn.on_click(select_team)
display(widgets.VBox(pos_dropdowns + ph_dropdowns + [select_btn, select_out]))

In [2]:
import ipywidgets as widgets
from IPython.display import display

def get_lineup_names():
    names = [enum_selected['C']['name'], enum_selected['1B']['name'],
             enum_selected['2B']['name'], enum_selected['3B']['name'],
             enum_selected['SS']['name']]
    names += [p['name'] for p in enum_selected['OF']]
    names.append(enum_selected['DH']['name'])
    return names

template_names = get_lineup_names()

# Lineup dropdowns
slot_dropdowns = [widgets.Dropdown(options=['Select'] + template_names, description=str(i+1)) for i in range(len(template_names))]
confirm_lineup_btn = widgets.Button(description='Set lineup')
lineup_out = widgets.Output()

def confirm_lineup(b):
    lineup_out.clear_output()
    picks = [d.value for d in slot_dropdowns]
    if 'Select' in picks:
        with lineup_out:
            print('Error: All lineup slots must be assigned.')
        return
    if len(set(picks)) != len(picks):
        with lineup_out:
            print('Error: Duplicate lineup assignment.')
        return
    # Assign lineup slots
    for i, name in enumerate(picks, start=1):
        for key, val in enum_selected.items():
            if key == 'OF':
                for p in val:
                    if p['name'] == name:
                        p['lineup_slot'] = i
            elif key not in ['PH'] and val.get('name') == name:
                val['lineup_slot'] = i
    with lineup_out:
        print('Final lineup:')
        print(enum_selected)

confirm_lineup_btn.on_click(confirm_lineup)
display(widgets.VBox(slot_dropdowns + [confirm_lineup_btn, lineup_out]))

# PH Order dropdowns
ph_order_dropdowns = [widgets.Dropdown(options=['Select'] + [p['name'] for p in enum_selected['PH']], description=f'PH{i+1}') for i in range(len(enum_selected['PH']))]
confirm_ph_order_btn = widgets.Button(description='Set PH order')
ph_order_out = widgets.Output()

def confirm_ph_order(b):
    ph_order_out.clear_output()
    picks = [d.value for d in ph_order_dropdowns]
    if 'Select' in picks:
        with ph_order_out:
            print('Error: All PH slots must be assigned.')
        return
    if len(set(picks)) != len(picks):
        with ph_order_out:
            print('Error: Duplicate PH assignment.')
        return
    # Assign bench order
    for i, name in enumerate(picks, start=1):
        for p in enum_selected['PH']:
            if p['name'] == name:
                p['bench_order'] = i
    with ph_order_out:
        print('Final PH order:')
        print(enum_selected['PH'])

confirm_ph_order_btn.on_click(confirm_ph_order)
display(widgets.VBox(ph_order_dropdowns + [confirm_ph_order_btn, ph_order_out]))


In [3]:
from collections import deque

class FantasyGame:
    def __init__(self):
        self.inning = 1
        self.outs = 0
        self.bases = {'1B': None, '2B': None, '3B': None}
        self.score = 0
        self.current_batter = None
        self.game_over = False
        self.log = []  # store play-by-play messages

    def _record(self, msg):
        print(msg)
        self.log.append(msg)

    def hit(self, bases_advanced):
        # Record a hit event
        self._record(f"{self.current_batter['name']} hits for {bases_advanced}-base hit.")
        # Advance runners
        # Prior runner on 2B always scores on single
        if bases_advanced == 1 and self.bases['2B']:
            self._record(f"{self.bases['2B']['name']} scores from second.")
            self.bases['2B'] = None
            self.score += 1

        for base in reversed(range(1, 4)):
            runner = self.bases[f'{base}B']
            if runner:
                new_base = base + bases_advanced
                if new_base > 3:
                    self._record(f"{runner['name']} scores.")
                    self.score += 1
                else:
                    self.bases[f'{new_base}B'] = runner
                self.bases[f'{base}B'] = None

        new_base = bases_advanced
        if new_base > 3:
            self._record(f"{self.current_batter['name']} scores on the hit.")
            self.score += 1
        else:
            self.bases[f'{new_base}B'] = self.current_batter

    def walk(self):
        self._record(f"{self.current_batter['name']} walks.")
        if self.bases['1B'] and self.bases['2B'] and self.bases['3B']:
            self._record(f"{self.bases['3B']['name']} scores on the walk.")
            self.score += 1
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
        elif self.bases['1B'] and self.bases['2B']:
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
        elif self.bases['1B']:
            self.bases['2B'] = self.bases['1B']
        self.bases['1B'] = self.current_batter

    def out(self):
        self.outs += 1
        self._record(f"Out recorded. Total outs: {self.outs}.")
        if self.outs >= 3:
            self._record(f"End of inning {self.inning}.")
            self.inning += 1
            self.outs = 0
            self.bases = {'1B': None, '2B': None, '3B': None}
            if self.inning > 9:
                self.game_over = True

    def double_play(self):
        self._record(f"{self.current_batter['name']} grounds into a double play.")
        # Simplified DP: two outs
        self.outs += 2
        if self.outs >= 3:
            self._record(f"End of inning {self.inning} after double play.")
            self.inning += 1
            self.outs = 0
            self.bases = {'1B': None, '2B': None, '3B': None}
            if self.inning > 9:
                self.game_over = True

    def process_at_bat(self, result):
        name = self.current_batter['name']
        self._record(f"{name} at bat: {result}.")
        if "Single" in result:
            self.hit(1)
        elif "Double" in result:
            self.hit(2)
        elif "Triple" in result:
            self.hit(3)
        elif "Home Run" in result:
            self.hit(4)
        elif "Walk" in result or "Hit By Pitch" in result:
            self.walk()
        elif "Double Play" in result:
            self.double_play()
        else:
            self.out()
        self._record(f"State: Inning {self.inning}, Outs {self.outs}, Bases {self.bases}, Score {self.score}")

# Example of how to run the simulation and then view the log:
game = FantasyGame()
# ... (run your simulation loop, setting game.current_batter and calling process_at_bat)

# After simulation, view full log:
for entry in game.log:
    print(entry)


In [6]:
# ────────────── Cell 4: Run Simulation & Print Play-by-Play ──────────────
from collections import deque

# Rebuild the batting order queue from selected_players
lineup_queue = deque(
    (pos, lineup_out[pos])
    for pos in ['C','1B','2B','3B','SS','LF','CF','RF','DH','SP','RP']
)

# And the PH (bench) queue
ph_queue = deque(selected_players['PH'])

# Track which at-bat sequence each batter last used
last_sequence = {info['playerID']: 0 for _, info in lineup_queue}
for ph in ph_queue:
    last_sequence[ph['playerID']] = 0

# Create a fresh fantasy game
game = FantasyGame()

# Step through batters until the game ends
while not game.game_over:
    if not lineup_queue:
        break
    for _ in range(len(lineup_queue)):
        pos, info = lineup_queue.popleft()
        pid = info['playerID']
        # grab their next play
        nxt = (
            plays_all_games
            .query("playerID == @pid and sequence_number > @last_sequence[@pid]")
            .sort_values("sequence_number", ascending=True)
        )
        if not nxt.empty:
            row = nxt.iloc[0]
            game.current_batter = info
            game.process_result(row['result'])
            last_sequence[pid] = row['sequence_number']
            lineup_queue.append((pos, info))
        elif ph_queue:
            sub = ph_queue.popleft()
            game.sub_in(pos, info, sub)
            last_sequence[sub['playerID']] = 0
            lineup_queue.append((pos, sub))

# Print out the complete log
print("Play-by-Play Log:")
for entry in game.log:
    print(entry)


TypeError: 'Output' object is not subscriptable